# 2. Unit tests and contract tests

**Time: about 15 minutes**

Before we test anything clever, we test the boring parts. The tools are plain Python, so they get plain `pytest` tests.
These tests need no API key, cost nothing, run in milliseconds, and give the same result every time.

## Run the tool tests

In [1]:
!pytest tests/test_tools.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- /Users/datasense/Desktop/agent-evals-ci/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/datasense/Desktop/agent-evals-ci
configfile: pytest.ini
plugins: platformdirs-4.14.5, deepeval-4.3.2, repeat-0.9.4, xdist-3.8.0, anyio-4.16.0, rerunfailures-16.7, asyncio-1.4.1.post1
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=function, asyncio_default_test_loop_scope=function
collected 14 items                                                             

tests/test_tools.py::test_search_finds_both_pizza_places PASSED          [  7%]
tests/test_tools.py::test_search_ignores_case_and_spaces[Pizza] PASSED   [ 14%]
tests/test_tools.py::test_search_ignores_case_and_spaces[PIZZA] PASSED   [ 21%]
tests/test_tools.py::test_search_ignores_case_and_spaces[ pizza ] PASSED [ 28%]
tests/test_tools.py::test_search_for_a_cuisine_

## What a good test looks like

Read a few of them. Each one:

* has a **name that says the behavior** (`test_closed_restaurant_cannot_take_orders`), so a failure explains itself;
* checks **one thing**;
* follows **arrange, act, assert**: set up the input, call the code, check the result.

In [2]:
from IPython.display import Code

Code(filename="tests/test_tools.py", language="python")

"""Unit tests for the tools. Plain Python, no AI, no API key: they give the same result every run."""

import json

import pytest

from tools import get_menu, place_order, search_restaurants, track_order


def test_search_finds_both_pizza_places():
    names = [r["name"] for r in json.loads(search_restaurants("pizza"))]
    assert names == ["Napoli Express", "Slice House"]


@pytest.mark.parametrize("cuisine", ["Pizza", "PIZZA", " pizza "])
def test_search_ignores_case_and_spaces(cuisine):
    assert len(json.loads(search_restaurants(cuisine))) == 2


def test_search_for_a_cuisine_we_do_not_serve_finds_nothing():
    assert json.loads(search_restaurants("sushi")) == []


def test_menu_has_prices():
    menu = json.loads(get_menu("Green Bowl"))["menu"]
    assert {"item": "Falafel Bowl", "price": 10.00, "vegetarian": True} in menu


def test_menu_says_when_a_restaurant_is_closed():
    assert json.loads(get_menu("Slice House"))["open"] is False


def test_menu_of_an_unknown_restaurant_is_empty():
    assert json.loads(get_menu("Burger Barn")) == {"restaurant": "Burger Barn", "open": False, "menu": []}


def test_open_restaurant_takes_orders():
    result = json.loads(place_order("Napoli Express", "Margherita Pizza"))
    assert result == {"order_id": "QB-2002", "restaurant": "Napoli Express", "item": "Margherita Pizza"}


@pytest.mark.parametrize(
    "restaurant, item, error",
    [
        ("Slice House", "Veggie Supreme", "Slice House is closed"),
        ("Burger Barn", "Cheeseburger", "Unknown restaurant: Burger Barn"),
        ("Napoli Express", "Falafel Bowl", "Falafel Bowl is not on the menu at Napoli Express"),
    ],
)
def test_bad_orders_are_refused(restaurant, item, error):
    assert json.loads(place_order(restaurant, item)) == {"error": error}


def test_track_known_order():
    assert json.loads(track_order("QB-1001"))["status"].startswith("Out for delivery")


def test_track_unknown_order():
    assert json.loads(track_order("QB-9999"))["status"] == "Order not found"

## One test, many inputs

`@pytest.mark.parametrize` runs the same test with different inputs.
That is how `test_bad_orders_are_refused` covers three different mistakes in five lines.
Use it whenever you catch yourself copying a test and changing one value.

## Test the unhappy paths

Most bugs live in the cases nobody planned for: a closed restaurant, an unknown restaurant, a dish that is not on the menu.

Here is a real story from this repo. The first version of `place_order` checked that the restaurant was open, and nothing else.
So this worked, and a pizza place happily took an order for a falafel bowl:

```python
place_order("Napoli Express", "Falafel Bowl")   # first version: {"order_id": "QB-2002", ...}
```

Writing the unhappy path test found it. Now the tool refuses:

In [3]:
from tools import place_order

print(place_order("Napoli Express", "Falafel Bowl"))
print(place_order("Burger Barn", "Cheeseburger"))

{"error": "Falafel Bowl is not on the menu at Napoli Express"}
{"error": "Unknown restaurant: Burger Barn"}


This is **defense in depth**. We will also tell the model not to do this, and evaluate that it does not.
But models make mistakes, so the tool itself refuses too. A rule enforced in code cannot be talked out of.

## Contract tests: keep the model's view in sync with the code

The model never sees `tools.py`. It sees `TOOL_SCHEMAS`. If the two drift apart, nothing crashes on your laptop,
but in production every tool call fails.

Let's write the check by hand first:

In [4]:
import copy
import inspect

from tools import TOOL_SCHEMAS, TOOLS


def schema_matches_function(schema):
    function = TOOLS[schema["function"]["name"]]
    return set(schema["function"]["parameters"]["properties"]) == set(inspect.signature(function).parameters)


print("Today:", [schema_matches_function(s) for s in TOOL_SCHEMAS])

# Someone renames a parameter in the schema, but not in the function:
broken = copy.deepcopy(TOOL_SCHEMAS[1])
broken["function"]["parameters"]["properties"] = {"restaurant_name": {"type": "string", "description": "Name"}}
print("After the rename:", schema_matches_function(broken))

Today: [True, True, True, True]
After the rename: False


And this is what the agent would see at runtime when the model follows the broken schema:

In [5]:
from agent import call_tool

print(call_tool("get_menu", {"restaurant_name": "Green Bowl"}))

{"error": "Bad arguments for get_menu: get_menu() got an unexpected keyword argument 'restaurant_name'"}


The agent does not crash (we will test that in notebook 3), but no customer can ever read a menu again.
A contract test catches this before the code leaves your laptop.

## The prompt is a contract too

The system prompt tells the model which cuisines exist. If we add a sushi restaurant but forget the prompt,
the model will tell customers we do not serve sushi. Let's simulate it, in memory only:

In [6]:
from agent import SYSTEM_PROMPT
from tools import RESTAURANTS

RESTAURANTS["Sushi Go"] = {"cuisine": "sushi", "open": True}
missing = [r["cuisine"] for r in RESTAURANTS.values() if r["cuisine"] not in SYSTEM_PROMPT]
print("Cuisines the model was never told about:", missing)

del RESTAURANTS["Sushi Go"]

Cuisines the model was never told about: ['sushi']


## Run the contract tests

In [7]:
Code(filename="tests/test_contracts.py", language="python")

"""Contract tests: the tool schemas the model sees must match the Python functions we actually run.

If someone renames a parameter in tools.py but forgets the schema, the model sends the old name and
every call fails at runtime. These tests catch that on the developer's laptop, for free.
"""

import inspect

import pytest

from agent import SYSTEM_PROMPT
from tools import RESTAURANTS, TOOL_SCHEMAS, TOOLS

SCHEMAS = {s["function"]["name"]: s["function"] for s in TOOL_SCHEMAS}


def test_every_tool_has_a_schema_and_every_schema_has_a_tool():
    assert set(SCHEMAS) == set(TOOLS)


@pytest.mark.parametrize("name", sorted(TOOLS))
def test_schema_parameters_match_the_function(name):
    function_params = set(inspect.signature(TOOLS[name]).parameters)
    schema = SCHEMAS[name]["parameters"]
    assert set(schema["properties"]) == function_params
    assert set(schema["required"]) == function_params


@pytest.mark.parametrize("name", sorted(TOOLS))
def test_every_tool_is_described_for_the_model(name):
    assert SCHEMAS[name]["description"].strip()
    assert all(p["description"].strip() for p in SCHEMAS[name]["parameters"]["properties"].values())


@pytest.mark.parametrize("cuisine", sorted({r["cuisine"] for r in RESTAURANTS.values()}))
def test_the_model_is_told_about_every_cuisine_we_serve(cuisine):
    assert cuisine in SYSTEM_PROMPT
    assert cuisine in SCHEMAS["search_restaurants"]["description"]

In [8]:
!pytest tests/test_contracts.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- /Users/datasense/Desktop/agent-evals-ci/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/datasense/Desktop/agent-evals-ci
configfile: pytest.ini
plugins: platformdirs-4.14.5, deepeval-4.3.2, repeat-0.9.4, xdist-3.8.0, anyio-4.16.0, rerunfailures-16.7, asyncio-1.4.1.post1
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=function, asyncio_default_test_loop_scope=function
collected 12 items                                                             

tests/test_contracts.py::test_every_tool_has_a_schema_and_every_schema_has_a_tool PASSED [  8%]
tests/test_contracts.py::test_schema_parameters_match_the_function[get_menu] PASSED [ 16%]
tests/test_contracts.py::test_schema_parameters_match_the_function[place_order] PASSED [ 25%]
tests/test_contracts.py::test_schema_parameters_match_the_function[search_restaurants]

## Takeaways

* Test every tool like normal code, especially the unhappy paths.
* Enforce hard rules in the tool, not only in the prompt.
* Write contract tests for anything the model sees: tool schemas, and facts in the prompt.

## Try it yourself

1. Add a fifth tool, `cancel_order(order_id)`, to `tools.py` but **not** to `TOOL_SCHEMAS`. Run `pytest tests/test_contracts.py`. What fails?
2. Write a unit test that `get_menu` works with the exact name only (what should `get_menu("green bowl")` return?). Decide what the right behavior is first, then write the test.
3. Add a test that every price on every menu is above zero.